# Weapon Detection — Baseline Error Analysis

**Purpose:** Diagnose *why* the Epoch-46 baseline fails and use the evidence to decide what should be improved next.

This notebook is intentionally separate from model training. It does **not** retrain the model or tune the locked Test set.

### Locked baseline
- Model: **YOLO26m**
- Input size: **640×640**
- Best validation checkpoint: `FINAL_CONTINUATION/checkpoints/best.pt`
- Validation mAP50-95: **0.493**
- Locked Test mAP50-95: **0.4111**
- Classes: `Knife`, `Machette`, `Handgun`, `Longgun`, `Other`

### Diagnostic philosophy
The official mAP metrics remain the benchmark. The error analysis below is an explanatory layer used to answer:
- What is being missed?
- What is being falsely detected?
- Which classes are involved?
- Are errors related to object scale, source/domain, blur, occlusion, or class confusion?
- Which failure modes should the next training experiment target?

> **Important:** The diagnostic FP/FN counts are produced with a fixed confidence/IoU rule and are not replacements for official mAP.

## 1. Environment and reproducibility

The finalized dataset is stored on Drive as a ZIP because copying thousands of individual files from mounted Drive is slow. For analysis, the ZIP is copied once to Colab local storage and extracted there.

Using local SSD is deliberate: validation/error inference should read images locally rather than repeatedly loading them through mounted Drive.

In [ ]:
import os
import shutil
import zipfile
import yaml
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

from google.colab import drive

# GPU is preferred, but the notebook can fall back to CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

## 2. Install and load the established baseline model

We use the **permanent Epoch-46 `best.pt`**. This model is frozen for this analysis; no weights are modified.

In [ ]:
!pip install -q ultralytics

MODEL_PATH = (
    "/content/drive/MyDrive/Weapon_Detection_Training/"
    "FINAL_CONTINUATION/checkpoints/best.pt"
)

assert os.path.isfile(MODEL_PATH), f"Model not found: {MODEL_PATH}"

model = YOLO(MODEL_PATH)
CLASS_NAMES = model.names

print("Model loaded successfully.")
print("Model:", MODEL_PATH)
print("Classes:", CLASS_NAMES)

## 3. Restore the finalized dataset locally

The Drive archive is the approved `Final_Dataset_Split.zip`. We copy/extract it only when needed, so a restarted runtime does not unnecessarily repeat the expensive transfer.

In [ ]:
DRIVE_ZIP = (
    "/content/drive/MyDrive/Final_Dataset_Split/"
    "Final_Dataset_Split.zip"
)

LOCAL_ZIP = "/content/Final_Dataset_Split.zip"
EXTRACT_DIR = "/content/WD_Dataset"

assert os.path.isfile(DRIVE_ZIP), f"Dataset ZIP not found: {DRIVE_ZIP}"

if not os.path.isfile(LOCAL_ZIP):
    print("Copying dataset ZIP to local Colab storage...")
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
else:
    print("Local ZIP already exists — skipping copy.")

if not os.path.isdir(EXTRACT_DIR):
    print("Extracting dataset...")
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        zf.extractall(EXTRACT_DIR)
else:
    print("Dataset already extracted — skipping extraction.")

# Locate the directory containing train/val/test.
DATASET_ROOT = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    if all(x in dirs for x in ("train", "val", "test")):
        DATASET_ROOT = root
        break

assert DATASET_ROOT is not None, "Could not find train/val/test dataset structure."

print("Dataset root:", DATASET_ROOT)

In [ ]:
# Create ONE analysis YAML with the local Colab dataset path.
DATA_YAML = "/content/wd_error_analysis.yaml"

dataset_config = {
    "path": DATASET_ROOT,
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 5,
    "names": {
        0: "Knife",
        1: "Machette",
        2: "Handgun",
        3: "Longgun",
        4: "Other",
    },
}

with open(DATA_YAML, "w") as f:
    yaml.dump(dataset_config, f, sort_keys=False)

VAL_IMAGES = os.path.join(DATASET_ROOT, "val", "images")
VAL_LABELS = os.path.join(DATASET_ROOT, "val", "labels")
TEST_IMAGES = os.path.join(DATASET_ROOT, "test", "images")
TEST_LABELS = os.path.join(DATASET_ROOT, "test", "labels")

assert os.path.isdir(VAL_IMAGES)
assert os.path.isdir(VAL_LABELS)
assert os.path.isdir(TEST_IMAGES)
assert os.path.isdir(TEST_LABELS)

def count_images(folder):
    return len([
        f for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ])

print("Validation images:", count_images(VAL_IMAGES))
print("Validation labels:", len([f for f in os.listdir(VAL_LABELS) if f.endswith(".txt")]))
print("Test images:", count_images(TEST_IMAGES))
print("Test labels:", len([f for f in os.listdir(TEST_LABELS) if f.endswith(".txt")]))
print("Analysis YAML:", DATA_YAML)

## 4. Baseline validation evaluation

This reproduces the official validation result on the finalized 2,305-image validation split.

We run it once here because the confusion matrix/plots are also useful for diagnosis. The official Test set is **not** used here.

In [ ]:
validation_results = model.val(
    data=DATA_YAML,
    split="val",
    imgsz=640,
    device=device,
    plots=True,
    verbose=True,
)

print("\nValidation evaluation completed.")

### Baseline validation result — Epoch 46

| Class | Precision | Recall | mAP50 | mAP50-95 |
|---|---:|---:|---:|---:|
| Knife | 0.639 | 0.547 | 0.526 | 0.377 |
| Machette | 0.602 | 0.700 | 0.676 | 0.476 |
| Handgun | 0.547 | 0.644 | 0.562 | 0.436 |
| Longgun | 0.931 | 0.701 | 0.764 | 0.548 |
| Other | 0.863 | 0.852 | 0.901 | 0.628 |
| **Overall** | **0.716** | **0.689** | **0.686** | **0.493** |

**Immediate signal:** Knife and Handgun are weak at class performance; Longgun has strong precision but still a substantial recall problem.

The locked Test benchmark remains **mAP50-95 = 0.4111**. It is not used for tuning.

## 5. Confusion matrix — what type of class errors occur?

This matrix is a diagnostic visualization. It is useful for identifying class confusion, but it should not be interpreted as a replacement for mAP.

In [ ]:
cm = validation_results.confusion_matrix.matrix

print("Confusion matrix shape:", cm.shape)
print("Class order:", CLASS_NAMES)

print("\nRaw matrix:")
print(cm)

# The plotted confusion matrix is also available in the Ultralytics
# validation output directory reported by model.val().

### Confusion-matrix finding

The main class-confusion pattern is **Knife ↔ Machette**.

In the diagnostic matrix, the important counts were:
- Machette → Knife: **55**
- Knife → Machette: **54**

Other classes had far fewer direct class confusions.

This means Knife/Machette needs a different kind of improvement from Longgun: the former is strongly affected by **fine-grained class separation**, while Longgun is dominated more by missed detections.

## 6. Individual error extraction

### Diagnostic settings
- Confidence threshold: **0.25**
- IoU matching threshold: **0.45**
- Inference size: **640**
- GPU used when available

The result is saved to Drive so the expensive inference does not need to be repeated after a runtime restart.

The matching is intentionally simple and diagnostic:
- matched same-class prediction → TP
- matched different-class prediction → class confusion
- unmatched ground truth → FN
- unmatched prediction → FP

**Caveat:** the saved table is a diagnostic record, not an official object-detection metric.

In [ ]:
ERROR_DIR = (
    "/content/drive/MyDrive/Weapon_Detection_Training/"
    "ERROR_ANALYSIS"
)
os.makedirs(ERROR_DIR, exist_ok=True)

ERROR_CSV = os.path.join(
    ERROR_DIR,
    "validation_error_cases.csv"
)

# Reuse an existing diagnostic table when available.
if os.path.isfile(ERROR_CSV):
    errors_df = pd.read_csv(ERROR_CSV)
    print("Existing error table loaded.")
    print("Rows:", len(errors_df))
else:
    CONF_THRESHOLD = 0.25
    IOU_THRESHOLD = 0.45
    IMG_SIZE = 640
    BATCH_SIZE = 32

    def load_labels(label_path):
        labels = []
        if not os.path.exists(label_path):
            return labels

        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, x, y, w, h = map(float, parts)
                    labels.append({
                        "cls": int(cls),
                        "box": [x, y, w, h],
                    })
        return labels

    def xywhn_to_xyxy(box, img_w, img_h):
        x, y, w, h = box
        return [
            (x - w / 2) * img_w,
            (y - h / 2) * img_h,
            (x + w / 2) * img_w,
            (y + h / 2) * img_h,
        ]

    def calculate_iou(box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])

        inter = max(0, x2 - x1) * max(0, y2 - y1)
        area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
        area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
        union = area1 + area2 - inter

        return inter / union if union > 0 else 0.0

    image_files = sorted([
        f for f in os.listdir(VAL_IMAGES)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ])

    error_cases = []

    print(f"Validation images: {len(image_files)}")
    print(f"Device: {device}")
    print(f"Confidence threshold: {CONF_THRESHOLD}")
    print(f"IoU threshold: {IOU_THRESHOLD}")

    for start in tqdm(
        range(0, len(image_files), BATCH_SIZE),
        desc="Validation inference",
    ):
        batch_files = image_files[start:start + BATCH_SIZE]
        batch_paths = [os.path.join(VAL_IMAGES, f) for f in batch_files]

        predictions = model.predict(
            source=batch_paths,
            imgsz=IMG_SIZE,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            device=device,
            verbose=False,
        )

        for image_file, result in zip(batch_files, predictions):
            img_h, img_w = result.orig_shape

            label_path = os.path.join(
                VAL_LABELS,
                os.path.splitext(image_file)[0] + ".txt",
            )
            gt_raw = load_labels(label_path)

            gt_boxes = [
                {
                    "cls": gt["cls"],
                    "box": xywhn_to_xyxy(gt["box"], img_w, img_h),
                }
                for gt in gt_raw
            ]

            pred_boxes = []
            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes.xyxy.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy().astype(int)
                confs = result.boxes.conf.cpu().numpy()

                for box, cls, conf in zip(boxes, classes, confs):
                    pred_boxes.append({
                        "cls": int(cls),
                        "box": box.tolist(),
                        "conf": float(conf),
                    })

            matched_gt = set()
            matched_pred = set()
            candidates = []

            for p_idx, pred in enumerate(pred_boxes):
                for g_idx, gt in enumerate(gt_boxes):
                    iou = calculate_iou(pred["box"], gt["box"])
                    if iou >= IOU_THRESHOLD:
                        candidates.append((iou, p_idx, g_idx))

            candidates.sort(reverse=True)

            for iou, p_idx, g_idx in candidates:
                if p_idx in matched_pred or g_idx in matched_gt:
                    continue

                matched_pred.add(p_idx)
                matched_gt.add(g_idx)

                pred_cls = pred_boxes[p_idx]["cls"]
                gt_cls = gt_boxes[g_idx]["cls"]

                if pred_cls != gt_cls:
                    error_cases.append({
                        "image": image_file,
                        "error_type": "CLASS_CONFUSION",
                        "gt_class": CLASS_NAMES[gt_cls],
                        "pred_class": CLASS_NAMES[pred_cls],
                        "confidence": pred_boxes[p_idx]["conf"],
                        "iou": float(iou),
                    })

            for g_idx, gt in enumerate(gt_boxes):
                if g_idx not in matched_gt:
                    error_cases.append({
                        "image": image_file,
                        "error_type": "FALSE_NEGATIVE",
                        "gt_class": CLASS_NAMES[gt["cls"]],
                        "pred_class": None,
                        "confidence": None,
                        "iou": 0.0,
                    })

            for p_idx, pred in enumerate(pred_boxes):
                if p_idx not in matched_pred:
                    error_cases.append({
                        "image": image_file,
                        "error_type": "FALSE_POSITIVE",
                        "gt_class": None,
                        "pred_class": CLASS_NAMES[pred["cls"]],
                        "confidence": pred["conf"],
                        "iou": 0.0,
                    })

    errors_df = pd.DataFrame(error_cases)
    errors_df.to_csv(ERROR_CSV, index=False)

    print("\nSaved:", ERROR_CSV)

print(errors_df["error_type"].value_counts())

### Diagnostic error totals

The completed validation error extraction produced:

- **690 false negatives**
- **522 false positives**
- **124 class confusions**
- **1,336 total diagnostic error cases**

These numbers explain where the model is struggling, but the official benchmark remains mAP/precision/recall.

## 7. Error breakdown by class

This distinguishes:
- classes that are mainly **missed**
- classes that are mainly **over-predicted**
- class pairs that are **confused**

In [ ]:
fn_df = errors_df[errors_df["error_type"] == "FALSE_NEGATIVE"]
fp_df = errors_df[errors_df["error_type"] == "FALSE_POSITIVE"]
conf_df = errors_df[errors_df["error_type"] == "CLASS_CONFUSION"].copy()

print("=" * 65)
print("FALSE NEGATIVES — MISSED WEAPONS")
print("=" * 65)
print(fn_df["gt_class"].value_counts().to_string())

print("\n" + "=" * 65)
print("FALSE POSITIVES — INCORRECT DETECTIONS")
print("=" * 65)
print(fp_df["pred_class"].value_counts().to_string())

print("\n" + "=" * 65)
print("CLASS CONFUSIONS")
print("=" * 65)
print(
    conf_df.groupby(["gt_class", "pred_class"])
    .size()
    .sort_values(ascending=False)
    .to_string()
)

### Class-level findings

**Longgun**
- 254 FN
- 37 FP
- Main problem: **recall/missed detections**, not over-prediction.

**Handgun**
- 181 FN
- 248 FP
- Main problem: **both missed detections and false positives**.

**Knife**
- 153 FN
- 101 FP
- 109 Knife/Machette cross-confusions.
- Main problem: **small objects plus difficult visual separation**.

**Machette**
- 59 FN
- 102 FP
- Strong Knife/Machette confusion.

**Other**
- comparatively strong and not the immediate priority.

## 8. Object-size analysis of false negatives

We test the hypothesis that missed weapons are disproportionately small.

Object size is measured as normalized bounding-box area (`width × height`). This is an explanatory analysis, not a new training metric.

In [ ]:
size_records = []

for _, row in fn_df.iterrows():
    image_name = row["image"]
    class_name = row["gt_class"]

    label_path = os.path.join(
        VAL_LABELS,
        os.path.splitext(image_name)[0] + ".txt",
    )

    if not os.path.exists(label_path):
        continue

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls, x, y, w, h = map(float, parts)

            if CLASS_NAMES[int(cls)] != class_name:
                continue

            area = w * h

            if area < 0.01:
                category = "Very Small (<1%)"
            elif area < 0.03:
                category = "Small (1–3%)"
            elif area < 0.08:
                category = "Medium (3–8%)"
            else:
                category = "Large (>8%)"

            size_records.append({
                "image": image_name,
                "class": class_name,
                "area": area,
                "size_category": category,
            })
            break

fn_size_df = pd.DataFrame(size_records)

print("=" * 65)
print("FALSE NEGATIVE OBJECT SIZE DISTRIBUTION")
print("=" * 65)
print(fn_size_df["size_category"].value_counts().to_string())

print("\n" + "=" * 65)
print("MEDIAN NORMALIZED OBJECT AREA BY CLASS")
print("=" * 65)
print(fn_size_df.groupby("class")["area"].median().sort_values().to_string())

### Size finding

From the 690 missed weapons:

- **334 (48.4%)** were very small (<1% of image area)
- **173 (25.1%)** were small (1–3%)
- Therefore **73.5% of FNs were ≤3% of image area**

Median missed-object area:
- Handgun: **0.109%**
- Knife: **0.687%**
- Machette: **0.728%**
- Longgun: **1.878%**
- Other: **7.65%**

This strongly suggests that **small-object detection is the dominant systemic weakness**.

## 9. Miss rate by object size

The previous cell looked only at FNs. This cell compares misses against the full ground-truth population so that size becomes a **miss-rate statistic**, not just an FN count.

In [ ]:
# Reconstruct the validation ground-truth object population.
# The error table is keyed by image/class, so this is a directional
# object-size analysis; repeated same-class objects in one frame
# are a known limitation of this diagnostic calculation.

missed_pairs = set(zip(fn_df["image"], fn_df["gt_class"]))

all_objects = []

for image_name in os.listdir(VAL_IMAGES):
    if not image_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
        continue

    label_path = os.path.join(
        VAL_LABELS,
        os.path.splitext(image_name)[0] + ".txt",
    )

    if not os.path.exists(label_path):
        continue

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls, x, y, w, h = map(float, parts)
            class_name = CLASS_NAMES[int(cls)]
            area = w * h

            if area < 0.01:
                category = "Very Small (<1%)"
            elif area < 0.03:
                category = "Small (1–3%)"
            elif area < 0.08:
                category = "Medium (3–8%)"
            else:
                category = "Large (>8%)"

            all_objects.append({
                "image": image_name,
                "class": class_name,
                "area": area,
                "size_category": category,
                "missed": (image_name, class_name) in missed_pairs,
            })

all_objects_df = pd.DataFrame(all_objects)

order = [
    "Very Small (<1%)",
    "Small (1–3%)",
    "Medium (3–8%)",
    "Large (>8%)",
]

size_summary = (
    all_objects_df
    .groupby("size_category")
    .agg(
        total_objects=("missed", "size"),
        missed_objects=("missed", "sum"),
    )
    .reindex(order)
)

size_summary["miss_rate_%"] = (
    size_summary["missed_objects"] /
    size_summary["total_objects"] * 100
)

print("=" * 65)
print("MISS RATE BY OBJECT SIZE")
print("=" * 65)
print(size_summary.round(2).to_string())

print("\n" + "=" * 65)
print("MISS RATE BY CLASS AND SIZE")
print("=" * 65)

class_size = (
    all_objects_df
    .groupby(["class", "size_category"])
    .agg(
        total_objects=("missed", "size"),
        missed_objects=("missed", "sum"),
    )
)

class_size["miss_rate_%"] = (
    class_size["missed_objects"] /
    class_size["total_objects"] * 100
)

print(class_size.round(2).to_string())

### Key quantitative finding

Overall miss rate increased sharply as weapons became smaller:

| Object size | Miss rate |
|---|---:|
| **Very Small (<1%)** | **54.50%** |
| Small (1–3%) | **39.87%** |
| Medium (3–8%) | **24.53%** |
| Large (>8%) | **14.93%** |

Class-specific examples:
- Very-small Handgun: **91.28% miss rate**
- Very-small Longgun: **71.83%**
- Small Knife: **52.69%**
- Large Machette: **4.35%**

**Inference:** scale is a primary cause of missed detection. Increasing resolution and/or creating a better small-object training distribution are evidence-based next steps.

## 10. Visual inspection of representative false negatives

The numerical analysis tells us *where* errors occur; representative images tell us *why*.

We inspect a small sample rather than hundreds of images.

In [ ]:
classes_to_inspect = ["Handgun", "Knife", "Longgun", "Machette"]
N_PER_CLASS = 6

selected = (
    fn_df[fn_df["gt_class"].isin(classes_to_inspect)]
    .groupby("gt_class", group_keys=False)
    .head(N_PER_CLASS)
)

fig, axes = plt.subplots(
    len(classes_to_inspect),
    N_PER_CLASS,
    figsize=(18, 12),
)

for row_idx, class_name in enumerate(classes_to_inspect):
    class_cases = selected[selected["gt_class"] == class_name].head(N_PER_CLASS)

    for col_idx in range(N_PER_CLASS):
        ax = axes[row_idx, col_idx]
        ax.axis("off")

        if col_idx >= len(class_cases):
            continue

        image_name = class_cases.iloc[col_idx]["image"]
        image_path = os.path.join(VAL_IMAGES, image_name)

        if os.path.exists(image_path):
            ax.imshow(Image.open(image_path).convert("RGB"))
            ax.set_title(f"{class_name}\n{image_name}", fontsize=8)

plt.tight_layout()
plt.savefig("/content/fn_contact_sheet.png", dpi=150, bbox_inches="tight")
plt.show()

### Visual findings

Representative missed examples showed distinct failure modes:

- **Handgun:** good frame quality but the handgun can be barely visible / only partly emerging from a pocket.
- **Knife:** frequently very small and/or blurred.
- **Longgun:** frame quality can be excellent, but multiple people/weapons and partial blocking create difficult cases.
- **Machette:** blur and thin-side presentation can make the weapon visually difficult.

**Inference:** the problem is not only object size. Blur, occlusion, viewpoint and partial visibility also contribute.

## 11. False-positive confidence analysis

If most false positives were low-confidence, changing the confidence threshold could be a simple remedy.

If many are high-confidence, threshold tuning alone cannot solve the underlying visual confusion.

In [ ]:
fp_df = errors_df[errors_df["error_type"] == "FALSE_POSITIVE"].copy()

print("=" * 65)
print("FALSE POSITIVE CONFIDENCE")
print("=" * 65)
print(fp_df["confidence"].describe().round(3).to_string())

fp_df["confidence_band"] = pd.cut(
    fp_df["confidence"],
    bins=[0.25, 0.40, 0.60, 0.80, 1.000001],
    labels=[
        "Low (0.25–0.40)",
        "Medium (0.40–0.60)",
        "High (0.60–0.80)",
        "Very High (0.80–1.00)",
        ],
    include_lowest=True,
)

print("\nFP confidence bands:")
print(fp_df["confidence_band"].value_counts().to_string())

print("\nFP confidence by predicted class:")
print(
    fp_df.groupby("pred_class")["confidence"]
    .agg(count="count", mean="mean", median="median", max="max")
    .sort_values("count", ascending=False)
    .round(3)
    .to_string()
)

### False-positive finding

There were **522 FPs**.

- Mean FP confidence: **0.560**
- **125 FPs (24%)** had confidence ≥0.80
- **191 FPs (37%)** had confidence ≥0.60

Handgun was the major FP source:
- **248 Handgun FPs**
- median confidence **0.766**

**Inference:** simply raising the confidence threshold is unlikely to solve the problem without sacrificing recall. The underlying visual/domain confusion needs attention.

## 12. High-confidence Handgun false positives

We inspect the highest-confidence Handgun FPs to identify the visual pattern behind the problem.

In [ ]:
handgun_fp = (
    errors_df[
        (errors_df["error_type"] == "FALSE_POSITIVE") &
        (errors_df["pred_class"] == "Handgun")
    ]
    .sort_values("confidence", ascending=False)
    .head(12)
)

print(handgun_fp[["image", "confidence"]].to_string(index=False))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax, (_, row) in zip(axes.flat, handgun_fp.iterrows()):
    ax.axis("off")
    image_name = row["image"]
    image_path = os.path.join(VAL_IMAGES, image_name)

    if os.path.exists(image_path):
        ax.imshow(Image.open(image_path).convert("RGB"))
        ax.set_title(
            f"Conf: {row['confidence']:.3f}\n{image_name}",
            fontsize=9,
        )

plt.tight_layout()
plt.savefig("/content/high_conf_handgun_fp.png", dpi=150, bbox_inches="tight")
plt.show()

### Visual finding

The high-confidence Handgun FPs revealed two recurring patterns:
- black/difficult visual objects that can resemble a handgun, including knife-on-dark-clothing scenes
- genuine negative frames where the model invents a handgun-like detection

The annotation check on the investigated CVAT/K20 examples did **not** indicate an obvious annotation error; empty labels were consistent with the frozen taxonomy when no weapon was visible.

**Inference:** this is primarily a visual generalization/hard-negative problem rather than evidence of a simple label-ID mistake.

## 13. Handgun FP source analysis

We now test whether the Handgun FP problem is concentrated in a particular data source/domain.

In [ ]:
def source_from_filename(filename):
    name = str(filename).upper()

    if "HGP" in name:
        return "HGP"
    if "KAGGLE" in name:
        return "KAGGLE"
    if "DANGEROUS" in name:
        return "DANGEROUS"
    if "CVAT" in name:
        return "CVAT"

    return "UNKNOWN"

handgun_fp_all = errors_df[
    (errors_df["error_type"] == "FALSE_POSITIVE") &
    (errors_df["pred_class"] == "Handgun")
].copy()

handgun_fp_all["source"] = handgun_fp_all["image"].apply(source_from_filename)

print("Handgun FP count by source:")
print(handgun_fp_all["source"].value_counts().to_string())

print("\nHandgun FP confidence by source:")
print(
    handgun_fp_all.groupby("source")["confidence"]
    .agg(count="count", mean="mean", median="median", max="max")
    .sort_values("count", ascending=False)
    .round(3)
    .to_string()
)

### Source finding

**209 / 248 (84.3%) Handgun FPs came from CVAT.**

CVAT Handgun-FP median confidence was **0.815**.

**Inference:** the Handgun FP problem is strongly associated with the CVAT source/domain rather than being evenly distributed across all sources.

## 14. Validation source/class distribution

To interpret the CVAT result, we compare the source composition of the validation split.

In [ ]:
records = []

for image_name in os.listdir(VAL_IMAGES):
    if not image_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
        continue

    source = source_from_filename(image_name)
    label_path = os.path.join(
        VAL_LABELS,
        os.path.splitext(image_name)[0] + ".txt",
    )

    lines = []
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

    if not lines:
        records.append({
            "image": image_name,
            "source": source,
            "class": "NEGATIVE",
        })
        continue

    for line in lines:
        parts = line.split()
        if len(parts) == 5:
            cls = int(float(parts[0]))
            records.append({
                "image": image_name,
                "source": source,
                "class": CLASS_NAMES[cls],
            })

distribution_df = pd.DataFrame(records)

image_source = distribution_df[["image", "source"]].drop_duplicates()

print("=" * 65)
print("VALIDATION IMAGES BY SOURCE")
print("=" * 65)
print(image_source["source"].value_counts().to_string())

print("\n" + "=" * 65)
print("LABELED OBJECTS BY SOURCE")
print("=" * 65)
print(pd.crosstab(distribution_df["source"], distribution_df["class"]).to_string())

negative_images = (
    distribution_df[distribution_df["class"] == "NEGATIVE"]
    [["image", "source"]]
    .drop_duplicates()
)

negative_counts = negative_images["source"].value_counts()
source_totals = image_source["source"].value_counts()

negative_pct = (
    negative_counts.div(source_totals).mul(100).fillna(0)
)

print("\n" + "=" * 65)
print("NEGATIVE IMAGE PERCENTAGE")
print("=" * 65)
print(negative_pct.round(2).sort_values(ascending=False).to_string())

### Source-distribution finding

Validation source composition:

- **CVAT:** 988 images; **56.38% negative**
- Dangerous: 967 images; 5.79% negative
- HGP: 248 images; 1.21% negative
- Kaggle: 102 images; 0% negative

CVAT also contains only **12 Machette objects**, much fewer than its Knife/Handgun/Longgun representation.

**Inference:** source/domain distribution is an important part of the error structure. CVAT negatives are not automatically "bad" and should not simply be removed; instead, hard-negative behavior and domain balance should be investigated.

## 15. What are CVAT Handgun FPs occurring on?

This separates two different failure modes:
1. hallucinated Handguns on genuinely weapon-free frames
2. Handgun predictions on frames containing another weapon, especially Knife

In [ ]:
cvat_handgun_fp = errors_df[
    (errors_df["error_type"] == "FALSE_POSITIVE") &
    (errors_df["pred_class"] == "Handgun") &
    (errors_df["image"].str.upper().str.contains("CVAT"))
].copy()

context_records = []

for image_name in cvat_handgun_fp["image"]:
    label_path = os.path.join(
        VAL_LABELS,
        os.path.splitext(image_name)[0] + ".txt",
    )

    gt_classes = []

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    gt_classes.append(CLASS_NAMES[int(float(parts[0]))])

    if not gt_classes:
        context = "NEGATIVE / NO WEAPON"
    elif "Handgun" in gt_classes:
        context = "Contains Handgun"
    elif "Knife" in gt_classes:
        context = "Contains Knife"
    elif "Machette" in gt_classes:
        context = "Contains Machette"
    elif "Longgun" in gt_classes:
        context = "Contains Longgun"
    elif "Other" in gt_classes:
        context = "Contains Other"
    else:
        context = "Other labelled content"

    context_records.append({
        "image": image_name,
        "context": context,
        "ground_truth": ", ".join(gt_classes),
    })

context_df = pd.DataFrame(context_records)

print("CVAT Handgun FPs:", len(context_df))
print("\nContext counts:")
print(context_df["context"].value_counts().to_string())

print("\nContext percentages:")
print(
    (context_df["context"].value_counts(normalize=True) * 100)
    .round(2)
    .to_string()
)

### CVAT Handgun FP context

Among **209 CVAT Handgun FPs**:

- **103 (49.28%)** were genuine negative/no-weapon frames
- **85 (40.67%)** contained a Knife
- 16 (7.66%) contained a Handgun
- 5 (2.39%) contained a Longgun

This confirms two major Handgun FP mechanisms:

1. **negative-frame hallucination**
2. **Knife → Handgun visual confusion**

The 16 "Contains Handgun" cases are best understood as duplicate/incorrect detections around frames that already contain a true Handgun; they are not equivalent to hallucinating a handgun in an empty frame.

## 16. Image-level binary weapon-presence evaluation

Object-detection metrics answer a detailed localization/classification question. For a product-level "is there any weapon in this frame?" capability, we also compute a separate **image-level binary metric** on the locked Test set.

Definitions:
- **TP:** weapon present and at least one weapon detection
- **TN:** no weapon present and no detection
- **FP:** no weapon present but a detection occurs
- **FN:** weapon present but no detection occurs

This is a supplementary metric. It does **not** replace the official Test mAP.

In [ ]:
BINARY_RESULTS_CSV = (
    "/content/drive/MyDrive/Weapon_Detection_Training/"
    "FINAL_CONTINUATION/test_evaluation/"
    "image_level_binary_results.csv"
)

# Reuse a saved result when available.
if os.path.isfile(BINARY_RESULTS_CSV):
    binary_df = pd.read_csv(BINARY_RESULTS_CSV)
    print("Saved image-level Test results loaded.")
else:
    CONF_THRESHOLD = 0.25

    test_files = sorted([
        f for f in os.listdir(TEST_IMAGES)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ])

    binary_records = []

    for image_name in tqdm(test_files, desc="Image-level Test inference"):
        image_path = os.path.join(TEST_IMAGES, image_name)
        label_path = os.path.join(
            TEST_LABELS,
            os.path.splitext(image_name)[0] + ".txt"
        )

        gt_weapon = False

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                gt_weapon = any(line.strip() for line in f)

        prediction = model.predict(
            source=image_path,
            imgsz=640,
            conf=CONF_THRESHOLD,
            device=device,
            verbose=False,
        )[0]

        pred_weapon = (
            prediction.boxes is not None
            and len(prediction.boxes) > 0
        )

        if gt_weapon and pred_weapon:
            outcome = "TP"
        elif not gt_weapon and not pred_weapon:
            outcome = "TN"
        elif not gt_weapon and pred_weapon:
            outcome = "FP"
        else:
            outcome = "FN"

        binary_records.append({
            "image": image_name,
            "ground_truth_weapon": gt_weapon,
            "predicted_weapon": pred_weapon,
            "result": outcome,
        })

    binary_df = pd.DataFrame(binary_records)
    binary_df.to_csv(BINARY_RESULTS_CSV, index=False)

TP = int((binary_df["result"] == "TP").sum())
TN = int((binary_df["result"] == "TN").sum())
FP = int((binary_df["result"] == "FP").sum())
FN = int((binary_df["result"] == "FN").sum())

TOTAL = len(binary_df)

accuracy = (TP + TN) / TOTAL
sensitivity = TP / (TP + FN) if TP + FN else 0
specificity = TN / (TN + FP) if TN + FP else 0
precision = TP / (TP + FP) if TP + FP else 0

print("=" * 65)
print("IMAGE-LEVEL BINARY WEAPON DETECTION — LOCKED TEST")
print("=" * 65)
print(f"Total images : {TOTAL}")
print(f"TP           : {TP}")
print(f"TN           : {TN}")
print(f"FP           : {FP}")
print(f"FN           : {FN}")
print("-" * 65)
print(f"Accuracy     : {accuracy * 100:.2f}%")
print(f"Sensitivity  : {sensitivity * 100:.2f}%")
print(f"Specificity  : {specificity * 100:.2f}%")
print(f"Precision    : {precision * 100:.2f}%")

### Locked Test image-level result

For the 784-image held-out Test set:

- TP = **320**
- TN = **231**
- FP = **6**
- FN = **227**
- **Image-level accuracy = 70.28%**
- Sensitivity/TPR = **58.50%**
- Specificity/TNR = **97.47%**
- Precision = **98.16%**

Plain-language interpretation:

> The baseline correctly determines whether a frame contains a weapon or not about **70% of the time** on unseen Test images, but its main weakness is **missing weapon-containing frames** rather than falsely flagging empty frames.

This is a **supplementary product-style metric** and should be reported separately from the official mAP benchmark.

# 17. Final baseline diagnosis

## Primary failure mode
**Small-object detection**

Evidence:
- Very-small objects (<1% of image area): **54.50% miss rate**
- Large objects (>8%): **14.93% miss rate**
- Very-small Handgun: **91.28% miss rate**
- Very-small Longgun: **71.83% miss rate**

## Secondary failure modes
**Blur / poor visibility**
- especially Knife and Machette

**Occlusion / difficult composition**
- especially Longgun and partially visible Handgun scenes

**Fine-grained class confusion**
- Knife ↔ Machette: **109 diagnostic confusions**

**Domain/source bias**
- **84.3% of Handgun FPs came from CVAT**
- CVAT had **56.38% negative frames**
- CVAT contained only **12 Machette objects**

## What this means for the next model

The next training stage should not be "more training" in the abstract. It should deliberately target:

1. **small-object robustness**
2. **cross-source/domain diversity**
3. **hard negatives for Handgun**
4. **Knife/Machette discrimination**
5. **difficult blur/occlusion/orientation examples**

Potential later deployment architecture:
- direct weapon detection for conventional surveillance
- person/ROI localization before weapon detection for drone/aerial scenes
- optional hierarchical weapon-presence → fine-grained classification pipeline

The current Epoch-46 `best.pt` should remain **frozen as Baseline V1** for all future comparisons.

# 18. Reproducibility / safety notes

- Do not modify the master `Final_Dataset_Split` in place.
- Keep the ZIP as the rapid Colab transfer artifact.
- Keep training data on local Colab SSD during GPU work.
- Keep the Epoch-46 checkpoint untouched.
- Do not repeatedly tune on the locked Test set.
- Save every future experiment under a unique name with its own checkpoint lineage.
- Use Validation for development decisions; reserve Test for controlled final comparison.

### Persistent analysis artifact

The main diagnostic file produced by this notebook is:

`Weapon_Detection_Training/ERROR_ANALYSIS/validation_error_cases.csv`

The image-level binary Test result is:

`Weapon_Detection_Training/FINAL_CONTINUATION/test_evaluation/image_level_binary_results.csv`